# 02 - Danh gia retrieval (V1/V2/V3) tren GPU

Dung QdrantClient(":memory:") + GPU T4 de embed cau hoi va rerank nhanh
hon nhieu so voi CPU local. Chay tren ca dev.jsonl va test.jsonl.
Output: `/kaggle/working/retrieval_results.json`.

In [ ]:
!pip install -q qdrant-client==1.19.1 FlagEmbedding==1.4.2

In [ ]:
import json

import pandas as pd
from qdrant_client import QdrantClient, models

IN_DIR = "/kaggle/input/vn-labor-law-chunks"

chunks = [json.loads(line) for line in open(f"{IN_DIR}/chunks.jsonl", encoding="utf-8")]
chunks_by_id = {c["id"]: c for c in chunks}
emb_df = pd.read_parquet(f"{IN_DIR}/embeddings.parquet")
dev_set = [json.loads(line) for line in open(f"{IN_DIR}/dev.jsonl", encoding="utf-8")]
test_set = [json.loads(line) for line in open(f"{IN_DIR}/test.jsonl", encoding="utf-8")]
print("chunks:", len(chunks), "dev:", len(dev_set), "test:", len(test_set))

In [ ]:
COLLECTION = "labor_law"
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=COLLECTION,
    vectors_config={"dense": models.VectorParams(size=1024, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams()},
)

points = []
for i, row in emb_df.iterrows():
    chunk = chunks_by_id[row["id"]]
    points.append(
        models.PointStruct(
            id=int(i),
            vector={
                "dense": list(row["dense"]),
                "sparse": models.SparseVector(
                    indices=[int(x) for x in row["sparse_indices"]],
                    values=[float(x) for x in row["sparse_values"]],
                ),
            },
            payload=chunk,
        )
    )
client.upsert(collection_name=COLLECTION, points=points)
print("da index", len(points), "chunk")

In [ ]:
from FlagEmbedding import BGEM3FlagModel, FlagReranker

embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)


def encode_query(text):
    out = embedder.encode([text], max_length=1024, return_dense=True, return_sparse=True)
    dense = out["dense_vecs"][0].tolist()
    sparse = {int(k): float(v) for k, v in out["lexical_weights"][0].items()}
    return dense, sparse


def search_dense(query, top_k=5):
    dense, _sparse = encode_query(query)
    res = client.query_points(collection_name=COLLECTION, query=dense, using="dense", limit=top_k)
    return [p.payload for p in res.points]


def search_hybrid(query, top_k=5, prefetch_limit=20):
    dense, sparse = encode_query(query)
    sparse_vector = models.SparseVector(indices=list(sparse.keys()), values=list(sparse.values()))
    res = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=dense, using="dense", limit=prefetch_limit),
            models.Prefetch(query=sparse_vector, using="sparse", limit=prefetch_limit),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k,
    )
    return [p.payload for p in res.points]


def search_reranked(query, top_k=5, prefetch_limit=20):
    candidates = search_hybrid(query, top_k=prefetch_limit, prefetch_limit=prefetch_limit)
    if not candidates:
        return []
    pairs = [[query, c["noi_dung"]] for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True, max_length=512)
    if isinstance(scores, float):
        scores = [scores]
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [c for c, _score in ranked[:top_k]]

In [ ]:
# Giong het eval/retrieval_metrics.py o local, giu nguyen dinh nghia de
# so sanh duoc voi ket qua da chay tren CPU.


def hit_at_k(retrieved_ids, relevant_ids, k):
    return any(rid in relevant_ids for rid in retrieved_ids[:k])


def mrr_at_k(retrieved_ids, relevant_ids, k):
    for rank, rid in enumerate(retrieved_ids[:k], start=1):
        if rid in relevant_ids:
            return 1.0 / rank
    return 0.0


def evaluate(dataset, search_fn, k_hit=5, k_mrr=10):
    hits, mrrs = [], []
    top_k = max(k_hit, k_mrr)
    for item in dataset:
        relevant = set(item.get("dieu_can_trich") or [])
        if not relevant:
            continue
        results = search_fn(item["question"], top_k)
        retrieved_ids = [r["id"] for r in results]
        hits.append(hit_at_k(retrieved_ids, relevant, k_hit))
        mrrs.append(mrr_at_k(retrieved_ids, relevant, k_mrr))
    return {
        "hit_at_k": sum(hits) / len(hits) if hits else 0.0,
        "mrr_at_k": sum(mrrs) / len(mrrs) if mrrs else 0.0,
        "n": len(hits),
    }

In [ ]:
import time

versions = {
    "V1_dense": lambda q, k: search_dense(q, top_k=k),
    "V2_hybrid_prefetch20": lambda q, k: search_hybrid(q, top_k=k, prefetch_limit=20),
    "V2_hybrid_prefetch50": lambda q, k: search_hybrid(q, top_k=k, prefetch_limit=50),
    "V3_rerank_top10": lambda q, k: search_reranked(q, top_k=k, prefetch_limit=10),
    "V3_rerank_top20": lambda q, k: search_reranked(q, top_k=k, prefetch_limit=20),
}

results = {}
for split_name, split_data in [("dev", dev_set), ("test", test_set)]:
    for name, fn in versions.items():
        t0 = time.time()
        metrics = evaluate(split_data, fn, k_hit=5, k_mrr=10)
        metrics["eval_time_s"] = round(time.time() - t0, 1)
        key = f"{name}__{split_name}"
        results[key] = metrics
        print(key, metrics)

with open("/kaggle/working/retrieval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("da luu ket qua")